# dots.tts 语音合成面板（小红书 · Colab 版）

一键启动**公网面板**：输入文字 → 选语言 → 传参考音频（可选）→ 出语音。

**模型缓存在你的 Google Drive**，下次启动不用重新下载 5GB。

**每次使用只需 3 步：**
1. 菜单「运行时 → 更改运行时类型 → GPU」
2. 跑「第 1 步」一键启动（首次约 5-8 分钟，之后快）
3. 打开打印出来的 `https://xxx.gradio.live` 公网地址

> ⚠️ 打开面板地址时要**开着梯子**（跟访问 Colab 同一个）。


## 第 0 步：确认 GPU（菜单操作，不是代码）

**运行时 → 更改运行时类型 → 硬件加速器选 GPU**，然后跑下面这格确认。


In [ ]:
!nvidia-smi


## 第 1 步：一键启动面板

跑这一格就行：自动挂载 Drive（缓存模型）→ 装环境（缺失才装）→ 启动面板 → 打印公网地址。


In [ ]:
import os, subprocess, time, re

PY = "/content/py311/bin/python"

# ---- 0. 挂载 Google Drive（缓存模型，下次免重下 5GB）----
CACHE = "/content/drive/MyDrive/dots_cache"
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    os.makedirs(CACHE, exist_ok=True)
    print("✅ Drive 已挂载，模型缓存：", CACHE)
except Exception as e:
    CACHE = None
    print("⚠️ Drive 未挂载（模型缓存在本地，下次需重下）：", e)

# ---- 1. 环境（缺失才重建，约3-5分钟）----
if not os.path.exists(PY):
    print("🔄 环境缺失，重建中（约3-5分钟）...")
    subprocess.run("pip install -q uv", shell=True)
    subprocess.run("uv python install 3.11", shell=True)
    subprocess.run("uv venv /content/py311 --python 3.11", shell=True)
    subprocess.run("uv pip install --python /content/py311/bin/python torch==2.11.0 torchaudio==2.11.0", shell=True)
    subprocess.run("uv pip install --python /content/py311/bin/python dots.tts huggingface_hub soundfile gradio", shell=True)
    print("✅ 环境重建完成")

# ---- 2. 写面板脚本 ----
panel_code = '''import gradio as gr
import soundfile as sf
import torch
from dots_tts.runtime import DotsTtsRuntime

cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
precision = "bfloat16" if cap[0] >= 8 else "float16"
print("加载模型...", flush=True)
runtime = DotsTtsRuntime.from_pretrained("dots-studio/dots.tts-soar", precision=precision, optimize=False)
print("模型加载完成", flush=True)

def synth(text, language, prompt_audio):
    prompt_path = None
    if prompt_audio is not None:
        sr, data = prompt_audio
        prompt_path = "/content/_tmp_ref.wav"
        sf.write(prompt_path, data, sr)
    res = runtime.generate(text=text, language=language or "auto_detect",
                           prompt_audio_path=prompt_path)
    return (res["sample_rate"], res["audio"].float().cpu().squeeze().numpy())

LANGS = ["auto_detect", "ZH", "EN", "Cantonese", "口音:粤语", "口音:四川话", "口音:东北话",
         "JA", "KO", "FR", "DE", "ES", "RU", "AR", "HI", "PT", "IT", "TH", "VI", "ID"]

demo = gr.Interface(
    fn=synth,
    inputs=[
        gr.Textbox(label="要合成的文字", lines=3, value="你好，欢迎使用小红书 dots.tts 语音合成。"),
        gr.Dropdown(LANGS, value="auto_detect", label="语言"),
        gr.Audio(label="参考音频（可选，上传 3-10 秒人声做声音克隆）", type="numpy"),
    ],
    outputs=gr.Audio(label="合成结果"),
    title="dots.tts 语音合成面板（小红书 HiLab）",
    description="输入文字 → 选语言 → 上传参考音频（可选）→ 提交，几秒后出语音。",
)
demo.launch(share=True, debug=False)
'''

open("panel.py", "w").write(panel_code)

# ---- 3. 启动面板 + 拿公网地址 ----
env = dict(os.environ)
if CACHE:
    env["HF_HOME"] = CACHE
subprocess.Popen([PY, "panel.py"], stdout=open("panel.log", "w"), stderr=subprocess.STDOUT, env=env)

url = None
for i in range(1, 361):
    time.sleep(1)
    if os.path.exists("panel.log"):
        m = re.search("https://[a-z0-9-]+.gradio.live", open("panel.log").read())
        if m:
            url = m.group(0)
            break
    if i % 30 == 0:
        print(f"  ... 已等 {i} 秒（首次需下 5GB 模型）", flush=True)

if url:
    open("panel_url.txt", "w").write(url)
    print("🌐 面板公网地址：", url)
    print("   用浏览器打开这个地址即可（保持梯子开启）。")
else:
    print("⚠️ 未获取到地址，日志：")
    print(open("panel.log").read()[-2000:] if os.path.exists("panel.log") else "无日志")


## 🔄 重启面板（会话没断、但面板挂了时用）

如果 Colab 还开着、只是面板打不开/地址失效，跑这格快速重启（**不重装环境**，只重新加载模型，约 1 分钟）。


In [ ]:
import subprocess, os, time, re

PY = "/content/py311/bin/python"
CACHE = "/content/drive/MyDrive/dots_cache"

# 杀掉旧面板进程
subprocess.run("pkill -f panel.py || true", shell=True)
time.sleep(2)

# 重新启动（环境已存在，不重装）
env = dict(os.environ)
if os.path.isdir(CACHE):
    env["HF_HOME"] = CACHE
subprocess.Popen([PY, "panel.py"], stdout=open("panel.log", "w"), stderr=subprocess.STDOUT, env=env)

url = None
for i in range(1, 301):
    time.sleep(1)
    if os.path.exists("panel.log"):
        m = re.search("https://[a-z0-9-]+.gradio.live", open("panel.log").read())
        if m:
            url = m.group(0)
            break
    if i % 30 == 0:
        print(f"  ... 已等 {i} 秒", flush=True)

if url:
    open("panel_url.txt", "w").write(url)
    print("🌐 新面板地址：", url)
else:
    print("⚠️ 失败，日志：")
    print(open("panel.log").read()[-2000:] if os.path.exists("panel.log") else "无日志")


## 📌 下次怎么用（重要）

**把本 notebook 保存到你的 Google Drive**，以后直接从 Drive 打开：

1. 菜单 **文件 → 在 Drive 中保存副本**
2. 下次用：从 Drive 打开这个副本 → 选 GPU → 跑「第 1 步」即可

**为什么快：**
- 模型缓存到了 Drive（`/content/drive/MyDrive/dots_cache`），**下次不重下 5GB**
- 只有 Python 环境需要重装（约 2-3 分钟），这是 Colab 免费版不可避免的

**地址有效期：** 面板地址只要 Colab 会话不断线就有效；断线重连后重跑「第 1 步」会拿到新地址。
